#### Audio Synthesis

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "parler-tts/parler-tts-mini-expresso"
model = AutoModelForSeq2SeqLM.from_pretrained(model_id, dtype="auto").to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)
set_seed(42)  # For reproducibility of results

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import soundfile as sf

def synthesize_dataset(csv_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path, index_col=0)
    audio_names = []
    for prompt_id, row in tqdm(df.iterrows(), total=len(df), desc="Generating Audio"):
        user_cmd = row["User_Command"]
        description = row["Voice_Description"]
        cmd_id = row["cmd_id"]
        # 1. Tokenize the acoustic description condition
        input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
        # 2. Tokenize the semantic text to be spoken
        prompt_input_ids = tokenizer(user_cmd, return_tensors="pt").input_ids.to(device)
        # 3. Generate the acoustic waveform tensor
        # We explicitly set prompt_input_ids to map the text directly to the description
        generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
        # Convert tensor to a numpy array
        audio_arr = generation.cpu().numpy().squeeze()
        # 4. Save to disk (Naming convention links it to the original command and variation)
        filename = f"prompt_{prompt_id}_cmd_{cmd_id}_varia_{prompt_id % 3}.wav"
        filepath = os.path.join(output_dir, f"prompt_{prompt_id}_cmd_{cmd_id}_varia_{prompt_id % 3}.wav")
        sf.write(filepath, audio_arr, model.config.sampling_rate)
        audio_names.append(filename)
    # 5. Save the updated dataframe with the audio paths
    df["audio_file_name"] = audio_names
    updated_csv_path = csv_path.replace(".csv", "_with_audio.csv")
    df.to_csv(updated_csv_path, index=False)

In [ ]:
synthesize_dataset("./data/cleaned_raw_train.csv", "./data/synthesized_train")
synthesize_dataset("./data/cleaned_raw_test.csv", "./data/synthesized_test")

#### Acoustic Quality and Variance Assessment